In [1]:
# WEEK 1
# importing necessary libraries & setting up th environment
import torch
import numpy as np
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'torch'

In [ ]:
# importing dataset
from datasets import load_dataset
data = load_dataset(
    "MultimodalUniverse/legacysurvey",
    split="train",
    streaming=True
)

In [2]:
print(data)

NameError: name 'data' is not defined

In [ ]:
# Dataset Exploration

# extracting sample from datset
try:
    sample = next(iter(data))
except StopIteration:
    raise RuntimeError("Dataset stream returned no samples")

# printing dataset & number of fields
print(data)
print("\nNumber of fields:", len(sample))
print("\nAvailable fields:")
for key in sample.keys():
    print(" -", key)

In [ ]:
# inspecting every field
print("\nField Information\n")
for key, value in sample.items():
    print(f"\n--- {key} ---")
    print("Type:", type(value))
    if hasattr(value, "shape"):
        print("Shape:", value.shape)
    if hasattr(value, "dtype"):
        print("Dtype:", value.dtype)

In [ ]:
# taking a sample imagse and inspecting the  mainimage
print("\nImage Information\n")
image = sample["image"]
print("Image type:", type(image))
print("Image:", image)
if hasattr(image, "shape"):
    print("Image shape:", image.shape)
if hasattr(image, "dtype"):
    print("Image dtype:", image.dtype)

In [ ]:
# inspecting the optical bands
print("\nOptical Bands\n")
bands = ["FLUX_G", "FLUX_R", "FLUX_I", "FLUX_Z"]

for band in bands:
    value = sample[band]
    print(f"\n{band}")
    print("Type:", type(value))
    print("Value:", value)
    if hasattr(value, "shape"):
        print("Shape:", value.shape)
    if hasattr(value, "dtype"):
        print("Dtype:", value.dtype)

In [ ]:
# Inspecting important metadata
print("\nGalaxy Metadata\n")
metadata_fields = [
    "object_id",
    "EBV",
    "SHAPE_R",
    "SHAPE_E1",
    "SHAPE_E2"
]
for field in metadata_fields:
    value = sample[field]
    print(f"\n{field}")
    print("Type:", type(value))
    print("Value:", value)

In [ ]:
# Inspecting RGB, mask and catalog
print("\nOther Important Fields")
for field in ["rgb", "object_mask", "catalog", "blobmodel"]:
    value = sample[field]
    print(f"\n--- {field} ---")
    print("Type:", type(value))
    if hasattr(value, "shape"):
        print("Shape:", value.shape)
    if hasattr(value, "dtype"):
        print("Dtype:", value.dtype)

In [ ]:
# Griz Visualization
image_data = sample["image"]

# Convert flux data to array
flux = np.array(image_data["flux"])

# Get band names
bands = image_data["band"]

print("Bands:", bands)
print("Flux shape:", flux.shape)

# Create 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, band_name, band_image in zip(
    axes.ravel(),
    bands,
    flux
):
    ax.imshow(band_image, cmap="gray")
    ax.set_title(f"{band_name}-band")
    ax.axis("off")

plt.suptitle("DESI Legacy Survey — g/r/i/z", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Loading pretrained DINOv2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
dinov2 = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)
dinov2 = dinov2.to(device)
dinov2.eval()

In [ ]:
# Image Preprocessing
# inspecting rgb image

rgb_image = sample["rgb"]
print("Original image type:", type(rgb_image))
print("Original image size:", rgb_image.size)
print("Original image mode:", rgb_image.mode)

# Defining DINOv2 preprocessing

from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Apply preprocessing
input_image = transform(rgb_image)

# Inspecting the resulting tensor
print("\nPreprocessed image:")
print("Type:", type(input_image))
print("Shape:", input_image.shape)
print("Dtype:", input_image.dtype)

In [ ]:
# Adding batch dimension
input_image = input_image.unsqueeze(0)
print("\nAfter adding batch dimension:")
print("Shape:", input_image.shape)

# Moving image to the same device as DINOv2
input_image = input_image.to(device)
print("Final device:", input_image.device)

In [ ]:
# Running DINOv2 Inference

# setting model in evaluation mode
# Disable gradient calculation
with torch.no_grad():
    # Passing the image
    output = dinov2(input_image)

# Inspecting the output
print("Output type:", type(output))
print("Output shape:", output.shape)

In [ ]:
# Extracting CLS token
# Extracting internal features from DINOv2
with torch.no_grad():
    features = dinov2.forward_features(input_image)

# Extracting the normalized CLS token
cls_token = features["x_norm_clstoken"]

# Printing the result
print("CLS token type:", type(cls_token))
print("CLS token shape:", cls_token.shape)
print("CLS token device:", cls_token.device)

In [ ]:
# Extracting patch tokens
patch_tokens = features["x_norm_patchtokens"]
print("Patch tokens type:", type(patch_tokens))
print("Patch tokens shape:", patch_tokens.shape)
print("Patch tokens device:", patch_tokens.device)

In [ ]:
# Inspecting feature dimensions

print("CLS token shape:")
print(cls_token.shape)
print("\nPatch tokens shape:")
print(patch_tokens.shape)
print("\nNumber of images:", patch_tokens.shape[0])
print("Number of patches:", patch_tokens.shape[1])
print("Embedding dimension:", patch_tokens.shape[2])

# Calculate patch grid
num_patches = patch_tokens.shape[1]
grid_size = int(num_patches ** 0.5)
print("\nPatch grid:", grid_size, "x", grid_size)
print("Total patches:", grid_size * grid_size)

In [ ]:
# Visualizing path grid

# Display the original preprocessed image
image_to_show = input_image[0].cpu().permute(1, 2, 0).numpy()

# Undo ImageNet normalization for visualization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

image_to_show = image_to_show * std + mean
image_to_show = np.clip(image_to_show, 0, 1)

plt.figure(figsize=(8, 8))
plt.imshow(image_to_show)

# Drawing the boundaires
patch_size = 14
image_size = input_image.shape[-1]

for i in range(0, image_size + 1, patch_size):
    plt.axhline(i, linewidth=0.5)
    plt.axvline(i, linewidth=0.5)

plt.title("DINOv2 ViT-S/14 — 16 × 16 Patch Grid")
plt.axis("off")
plt.show()

In [ ]:
# Initial Embedding exporation

print("CLS token:")
print(cls_token)

print("\nFirst patch token:")
print(patch_tokens[0, 0])

print("\n--------------------------------")

print("CLS token statistics:")
print("Minimum:", cls_token.min().item())
print("Maximum:", cls_token.max().item())
print("Mean:", cls_token.mean().item())
print("Standard deviation:", cls_token.std().item())

print("\nPatch token statistics:")
print("Minimum:", patch_tokens.min().item())
print("Maximum:", patch_tokens.max().item())
print("Mean:", patch_tokens.mean().item())
print("Standard deviation:", patch_tokens.std().item())